# 04 — RGCN Training

Train the 2-layer RGCN on the full heterogeneous graph. Compare accuracy to the GCN baseline.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import load_processed
from src.graph_builder import build_hetero_data
from src.models import (
    RGCNLinkPredictor, RELATION_MAP, NODE_TYPES_ORDER, flatten_hetero_graph
)
from src.training import train_rgcn, sample_negatives
from src.evaluation import compute_link_scores, compute_metrics, compute_curves, compute_ranking_metrics
from src.plotting import plot_training_curves, plot_roc_curves, plot_pr_curves
from src.utils import (
    set_seed, get_device, save_checkpoint, save_metrics, load_metrics,
    DATA_SPLITS, RESULTS_DIR, BLOG_ASSETS
)

%matplotlib inline

In [ ]:
SEED = 42
set_seed(SEED)
device = get_device()
print(f"Device: {device}")

# Load processed data
id_maps, edges, stats = load_processed()
hetero_data = build_hetero_data(id_maps, edges)
num_drugs = len(id_maps['drug'])
print(hetero_data)

# Flatten for RGCN
flat_ei, flat_et, offsets = flatten_hetero_graph(hetero_data, NODE_TYPES_ORDER, RELATION_MAP)
print(f"\nFlat graph: {flat_ei.shape[1]:,} total edges, {len(RELATION_MAP)} relation types")

# Load DDI splits
train_edges = torch.load(os.path.join(DATA_SPLITS, 'train_edges.pt'), weights_only=True)
val_edges = torch.load(os.path.join(DATA_SPLITS, 'val_edges.pt'), weights_only=True)
test_edges = torch.load(os.path.join(DATA_SPLITS, 'test_edges.pt'), weights_only=True)
train_edges_ud = torch.load(os.path.join(DATA_SPLITS, 'train_edges_ud.pt'), weights_only=True)

## 1. Build Training Graph

The RGCN trains on the full heterogeneous graph, but the DDI edges used for message passing are only the training set. We rebuild the flat graph with train DDI edges only.

In [ ]:
# Rebuild hetero_data with only training DDI edges for message passing
import copy
from torch_geometric.data import HeteroData

train_hetero = HeteroData()
train_hetero['drug'].num_nodes = hetero_data['drug'].num_nodes
train_hetero['gene'].num_nodes = hetero_data['gene'].num_nodes
train_hetero['disease'].num_nodes = hetero_data['disease'].num_nodes

# Use only train DDI edges for message passing
train_hetero['drug', 'interacts', 'drug'].edge_index = train_edges_ud.long()

# Keep all auxiliary edges
for key in hetero_data.edge_types:
    if key != ('drug', 'interacts', 'drug'):
        train_hetero[key].edge_index = hetero_data[key].edge_index

train_flat_ei, train_flat_et, _ = flatten_hetero_graph(
    train_hetero, NODE_TYPES_ORDER, RELATION_MAP
)
print(f"Training graph: {train_flat_ei.shape[1]:,} edges")

## 2. Train RGCN

In [ ]:
set_seed(SEED)

num_nodes_dict = {ntype: len(id_maps[ntype]) for ntype in NODE_TYPES_ORDER}
rgcn = RGCNLinkPredictor(
    num_nodes_dict, embed_dim=64,
    num_relations=len(RELATION_MAP), num_bases=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(rgcn.parameters(), lr=0.01)

rgcn_history, rgcn_best_auroc = train_rgcn(
    rgcn, optimizer,
    train_ei=train_edges,
    val_ei=val_edges,
    graph_ei=train_flat_ei,
    graph_et=train_flat_et,
    num_drugs=num_drugs,
    epochs=200,
    patience=20,
    neg_ratio=5,
    device=device,
)
print(f"\nBest validation AUROC: {rgcn_best_auroc:.4f}")
save_checkpoint(rgcn, f'rgcn_base_s{SEED}')

## 3. RGCN Evaluation

In [ ]:
rgcn.eval()

set_seed(SEED)
neg_test = sample_negatives(test_edges, num_drugs, test_edges.shape[1] * 5)

pos_scores = compute_link_scores(rgcn, test_edges, model_type='rgcn',
                                  graph_edge_index=train_flat_ei, graph_edge_type=train_flat_et,
                                  device=device)
neg_scores = compute_link_scores(rgcn, neg_test, model_type='rgcn',
                                  graph_edge_index=train_flat_ei, graph_edge_type=train_flat_et,
                                  device=device)

rgcn_metrics = compute_metrics(pos_scores, neg_scores)
rgcn_ranking = compute_ranking_metrics(
    rgcn, test_edges, num_drugs, model_type='rgcn',
    graph_edge_index=train_flat_ei, graph_edge_type=train_flat_et,
    device=device
)
rgcn_metrics.update(rgcn_ranking)
rgcn_curves = compute_curves(pos_scores, neg_scores)

print("RGCN Test Metrics:")
for k, v in rgcn_metrics.items():
    print(f"  {k}: {v:.4f}")

save_metrics(f'rgcn_base_s{SEED}', rgcn_metrics)

## 4. Compare All Models

In [ ]:
# Load GCN results
gcn_metrics = load_metrics(f'gcn_base_s{SEED}')

with open(os.path.join(RESULTS_DIR, 'curves_data.json'), 'r') as f:
    prev_curves = json.load(f)

# Add RGCN curves
all_curves = prev_curves.copy()
all_curves['rgcn'] = {k: v.tolist() if hasattr(v, 'tolist') else v for k, v in rgcn_curves.items()}

all_aurocs = {'heuristic': None, 'gcn': gcn_metrics['auroc'], 'rgcn': rgcn_metrics['auroc']}
all_auprcs = {'gcn': gcn_metrics['auprc'], 'rgcn': rgcn_metrics['auprc']}

# Convert curve arrays back to numpy
curves_np = {}
for name, c in all_curves.items():
    curves_np[name] = {k: np.array(v) for k, v in c.items()}

fig = plot_roc_curves(curves_np, all_aurocs)
plt.show()

# PR curves (skip heuristic—its scores aren't calibrated probabilities)
pr_curves = {k: v for k, v in curves_np.items() if k != 'heuristic'}
fig = plot_pr_curves(pr_curves, all_auprcs)
plt.show()

print("\n=== Model Comparison ===")
print(f"{'Metric':<15} {'GCN':>10} {'RGCN':>10}")
print("-" * 37)
for m in ['auroc', 'auprc', 'mrr', 'hits@10', 'hits@20', 'hits@50']:
    g = gcn_metrics.get(m, 0)
    r = rgcn_metrics.get(m, 0)
    print(f"{m:<15} {g:>10.4f} {r:>10.4f}")

## 5. Save Updated Curves

In [ ]:
with open(os.path.join(RESULTS_DIR, 'curves_data.json'), 'w') as f:
    json.dump(all_curves, f)

# Save training histories
with open(os.path.join(RESULTS_DIR, 'rgcn_history.json'), 'w') as f:
    json.dump(rgcn_history, f)

print("All results saved.")